# AnimalCLEF 2026 — Feature Visualization Notebook

Visualizes:
1. **Sample images** from each species
2. **Keypoints** (SIFT, SuperPoint, ALIKED, DISK) with scale/orientation  
3. **Match pairs** — same individual vs different individual with match lines
4. **Cached match score matrices** — heatmaps from V3 cache
5. **Score distributions** — positive (same ID) vs negative (different ID) pairs
6. **Embedding space** — TSNE of MegaDescriptor features colored by identity
7. **Calibration curves** — isotonic regression fitted in V3

> **No GPU required for most cells** — cells 1–4 use cached data only.
> Cells 5–6 (live keypoint extraction + matching) need GPU or CPU (~slow on CPU).


In [ ]:
# Install only what's needed if running fresh
import subprocess, sys
def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import cv2
except ImportError:
    pip_install("opencv-python")

try:
    import umap
except ImportError:
    pip_install("umap-learn")

try:
    import matplotlib
except ImportError:
    pip_install("matplotlib")

print("Imports ready!")


In [ ]:
import os
import sys
import json
import pickle
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyArrowPatch
import cv2
from PIL import Image
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

# ── Paths ──────────────────────────────────────────────────────────────────
ROOT       = "/Users/vaatsav/Desktop/New/AnimalCLEF_26"
DATA_ROOT  = os.path.join(ROOT, "data/raw")
IMG_ROOT   = os.path.join(DATA_ROOT, "images")       # images/{dataset}/{split}/{id}/*.jpg
META_CSV   = os.path.join(DATA_ROOT, "metadata.csv")
V3_CACHE   = os.path.join(ROOT, "FINAL_SOLUTION_v3/cache")
OUT_DIR    = os.path.join(ROOT, "FINAL_SOLUTION_v4/viz_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ── Load metadata ─────────────────────────────────────────────────────────
meta = pd.read_csv(META_CSV)
meta["full_path"] = meta["path"].apply(lambda p: os.path.join(DATA_ROOT, p))

print(f"Total images : {len(meta)}")
print(meta.groupby(["dataset","split"])["image_id"].count().to_string())
print()
print("Columns:", meta.columns.tolist())


## Section 1: Image Browser

Random samples from each species — just to get a feel for the data.


In [ ]:
np.random.seed(42)

DATASETS = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022", "TexasHornedLizards"]
COLORS   = ["#E07B39", "#4A90D9", "#50C878", "#C44D58"]

# Pick 3 random train images per species (use test for TexasHorned which has no train labels)
fig, axes = plt.subplots(len(DATASETS), 3, figsize=(12, 4*len(DATASETS)))
fig.suptitle("Sample Images per Species", fontsize=16, fontweight="bold")

for row, (ds, col) in enumerate(zip(DATASETS, COLORS)):
    split = "train" if ds != "TexasHornedLizards" else "test"
    df_sub = meta[(meta["dataset"] == ds) & (meta["split"] == split)]
    samples = df_sub.sample(min(3, len(df_sub)), random_state=42)
    
    for col_idx, (_, row_data) in enumerate(samples.iterrows()):
        ax = axes[row, col_idx]
        img = Image.open(row_data["full_path"]).convert("RGB")
        ax.imshow(img)
        ax.axis("off")
        id_label = str(row_data["identity"]).split("_")[-1] if pd.notna(row_data["identity"]) else "?"
        ax.set_title(f"ID: {id_label}\n{img.size[0]}×{img.size[1]}", fontsize=9)
    
    # Label the row
    axes[row, 0].set_ylabel(ds.replace("ID2025","\nID2025").replace("ID2022","\nID2022"),
                            fontsize=11, fontweight="bold", labelpad=10, color=col)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/01_image_browser.png", bbox_inches="tight", dpi=150)
plt.show()
print("Saved: 01_image_browser.png")


## Section 2: Cached Match Score Matrices

V3 computed pairwise match scores for its internal validation split (identity-preserving).
Each matrix is (N_val × N_val) with values in [0, 1].

**What to look for:**
- Block-diagonal structure → same individual = high scores
- Off-diagonal near-zero → different individuals = low scores
- Differences between SIFT vs SuperPoint vs ALIKED vs DISK


In [ ]:
EXTRACTORS = ["sift", "superpoint", "disk", "aliked"]
DATASET_NAMES = ["LynxID2025", "SalamanderID2025", "SeaTurtleID2022"]

for ds in DATASET_NAMES:
    # Load all 4 extractor matrices
    matrices = {}
    for ext in EXTRACTORS:
        fpath = f"{V3_CACHE}/matches/{ds}_val_{ext}.npy"
        if os.path.exists(fpath):
            matrices[ext] = np.load(fpath)
    
    if not matrices:
        print(f"No cached matrices for {ds}, skipping")
        continue
    
    n = list(matrices.values())[0].shape[0]
    print(f"\n{ds}: val set has {n} images")
    print(f"  Available extractors: {list(matrices.keys())}")
    
    # For heatmap: subsample to 200 images max (keeps image manageable)
    N_SHOW = min(200, n)
    idx = np.arange(N_SHOW)
    
    n_ext = len(matrices)
    fig, axes = plt.subplots(1, n_ext, figsize=(5*n_ext, 5))
    if n_ext == 1: axes = [axes]
    fig.suptitle(f"{ds} — Match Score Heatmaps (first {N_SHOW} val images)", fontsize=13, fontweight="bold")
    
    for ax, (ext_name, mat) in zip(axes, matrices.items()):
        sub = mat[:N_SHOW, :N_SHOW]
        im = ax.imshow(sub, cmap="hot", vmin=0, vmax=sub.max(), aspect="auto")
        ax.set_title(f"{ext_name.upper()}\nmax={mat.max():.3f}  mean={mat.mean():.4f}", fontsize=11)
        ax.set_xlabel("Val image index"); ax.set_ylabel("Val image index")
        plt.colorbar(im, ax=ax, fraction=0.046)
    
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/02_{ds}_heatmaps.png", bbox_inches="tight", dpi=120)
    plt.show()
    print(f"  Saved: 02_{ds}_heatmaps.png")


## Section 3: Score Distributions — Same vs Different Individual

For validation images that have identity labels, we can split pairs into:
- **Positive pairs**: Same individual (should have HIGH score)
- **Negative pairs**: Different individuals (should have LOW score)

**Good extractor** = clear separation between the two distributions.


In [ ]:
# V3's val split was a 20% identity-preserving split from the train set.
# We need to reconstruct which images went into val.
# The train embeddings shape + val shape = total train rows in metadata.
# V3 used identity-preserving split so we can approximate by taking last 20% of each identity.

def get_v3_val_labels(dataset_name):
    \"\"\"
    Reconstruct approximate identity labels for V3's internal val split.
    V3 used sklearn train_test_split with test_size=0.2, stratify=identity.
    We re-create this split to get the labels.
    Singletons (identity count == 1) are excluded from stratified split.
    \"\"\"
    from sklearn.model_selection import train_test_split

    df = meta[(meta["dataset"] == dataset_name) &
              (meta["split"] == "train") &
              (meta["identity"].notna())].copy()

    n_val = np.load(f"{V3_CACHE}/matches/{dataset_name}_val_sift.npy").shape[0]

    # Filter out singleton identities — can't stratify with 1 sample per class
    counts = df["identity"].value_counts()
    valid_ids = counts[counts >= 2].index
    df_strat = df[df["identity"].isin(valid_ids)].reset_index(drop=True)

    n_singletons = len(df) - len(df_strat)
    if n_singletons > 0:
        print(f"  Excluded {n_singletons} singleton-identity images from stratified split")

    # Do the same split V3 did (on stratifiable subset)
    train_idx, val_idx = train_test_split(
        np.arange(len(df_strat)), test_size=0.2, random_state=42,
        stratify=df_strat["identity"].values
    )

    val_df = df_strat.iloc[val_idx].reset_index(drop=True)

    if len(val_df) != n_val:
        print(f"  WARNING: expected {n_val} val images, got {len(val_df)}")

    return val_df["identity"].values, val_df["full_path"].values

for ds in DATASET_NAMES:
    fpath = f"{V3_CACHE}/matches/{ds}_val_sift.npy"
    if not os.path.exists(fpath):
        continue
    
    try:
        labels, paths = get_v3_val_labels(ds)
    except Exception as e:
        print(f"{ds}: could not reconstruct labels ({e}), skipping")
        continue
    
    n = len(labels)
    print(f"\n{ds}: {n} val images, {len(np.unique(labels))} identities")
    
    # Build pairwise same/different mask (upper triangle)
    same_mask = (labels[:, None] == labels[None, :])
    upper = np.triu(np.ones((n, n), dtype=bool), k=1)  # exclude diagonal
    pos_idx = np.where(same_mask & upper)
    neg_idx = np.where(~same_mask & upper)
    
    # Sample for speed
    MAX_PAIRS = 50_000
    if len(pos_idx[0]) > MAX_PAIRS:
        sel = np.random.choice(len(pos_idx[0]), MAX_PAIRS, replace=False)
        pos_idx = (pos_idx[0][sel], pos_idx[1][sel])
    if len(neg_idx[0]) > MAX_PAIRS:
        sel = np.random.choice(len(neg_idx[0]), MAX_PAIRS, replace=False)
        neg_idx = (neg_idx[0][sel], neg_idx[1][sel])
    
    print(f"  Positive pairs: {len(pos_idx[0])}, Negative pairs: {len(neg_idx[0])}")
    
    fig, axes = plt.subplots(1, len(EXTRACTORS), figsize=(5*len(EXTRACTORS), 4))
    fig.suptitle(f"{ds} — Score Distributions (Same ID vs Different ID)", fontsize=13, fontweight="bold")
    
    for ax, ext in zip(axes, EXTRACTORS):
        fpath = f"{V3_CACHE}/matches/{ds}_val_{ext}.npy"
        if not os.path.exists(fpath):
            ax.set_title(f"{ext} (no cache)"); continue
        
        mat = np.load(fpath)
        pos_scores = mat[pos_idx]
        neg_scores = mat[neg_idx]
        
        # Compute overlap area (AUC proxy)
        from sklearn.metrics import roc_auc_score
        y_true = np.concatenate([np.ones(len(pos_scores)), np.zeros(len(neg_scores))])
        y_score = np.concatenate([pos_scores, neg_scores])
        auc = roc_auc_score(y_true, y_score)
        
        bins = np.linspace(0, max(mat.max(), 0.01), 60)
        ax.hist(neg_scores, bins=bins, alpha=0.6, color="#E07B39", density=True, label="Diff ID")
        ax.hist(pos_scores, bins=bins, alpha=0.6, color="#4A90D9", density=True, label="Same ID")
        ax.set_title(f"{ext.upper()}\nAUC={auc:.3f}", fontsize=11)
        ax.set_xlabel("Match Score"); ax.set_ylabel("Density")
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)
        
        print(f"  {ext}: AUC={auc:.3f}  pos_mean={pos_scores.mean():.4f}  neg_mean={neg_scores.mean():.4f}")
    
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/03_{ds}_score_distributions.png", bbox_inches="tight", dpi=120)
    plt.show()


## Section 4: MegaDescriptor Embedding Space (t-SNE)

Projects 1536-dim MegaDescriptor embeddings to 2D using t-SNE.
Colors = identity. 

**Good embedding space** = same-identity images cluster together, different identities are separated.


In [ ]:
for ds in DATASET_NAMES:
    emb_path = f"{V3_CACHE}/global/{ds}_val_megadesc.npy"
    if not os.path.exists(emb_path):
        print(f"No val embeddings for {ds}"); continue
    
    embeddings = np.load(emb_path)  # (N, 1536)
    
    try:
        labels, paths = get_v3_val_labels(ds)
    except Exception as e:
        print(f"{ds}: could not reconstruct labels ({e}), skipping"); continue
    
    n = min(len(labels), embeddings.shape[0])
    embeddings = embeddings[:n]
    labels = labels[:n]
    
    # Subsample for speed if large
    MAX_VIZ = 800
    if n > MAX_VIZ:
        idx = np.random.choice(n, MAX_VIZ, replace=False)
        emb_sub = embeddings[idx]
        lab_sub = labels[idx]
        n_sub = MAX_VIZ
    else:
        emb_sub = embeddings
        lab_sub = labels
        n_sub = n
    
    print(f"\n{ds}: t-SNE on {n_sub} images, {len(np.unique(lab_sub))} identities...")
    
    # PCA first to 50 dims (faster t-SNE)
    from sklearn.decomposition import PCA
    n_pca = min(50, emb_sub.shape[1], n_sub - 1)
    pca = PCA(n_components=n_pca, random_state=42)
    emb_pca = pca.fit_transform(emb_sub)
    
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, n_sub//3), n_jobs=-1)
    emb_2d = tsne.fit_transform(emb_pca)
    
    # Encode labels as integers for coloring
    le = LabelEncoder()
    lab_int = le.fit_transform(lab_sub)
    n_ids = len(np.unique(lab_int))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"{ds} — MegaDescriptor Embedding Space (t-SNE, n={n_sub})", fontsize=13, fontweight="bold")
    
    # Left: colored by identity
    cmap = plt.cm.get_cmap("tab20" if n_ids <= 20 else "hsv", n_ids)
    sc = axes[0].scatter(emb_2d[:,0], emb_2d[:,1], c=lab_int, cmap=cmap, s=12, alpha=0.7)
    axes[0].set_title(f"Colored by Identity ({n_ids} IDs)", fontsize=11)
    axes[0].set_xlabel("t-SNE dim 1"); axes[0].set_ylabel("t-SNE dim 2")
    axes[0].grid(alpha=0.2)
    
    # Right: highlight top-5 identities with most images
    from collections import Counter
    top5 = [id_ for id_, _ in Counter(lab_sub).most_common(5)]
    colors_top = plt.cm.Set1(np.linspace(0, 1, 5))
    axes[1].scatter(emb_2d[:,0], emb_2d[:,1], c="lightgray", s=8, alpha=0.4, zorder=1)
    for i, id_ in enumerate(top5):
        mask = (lab_sub == id_)
        axes[1].scatter(emb_2d[mask,0], emb_2d[mask,1], c=[colors_top[i]],
                        s=25, alpha=0.9, label=str(id_).split("_")[-1], zorder=2)
    axes[1].legend(title="Top-5 IDs (by count)", fontsize=8, title_fontsize=9)
    axes[1].set_title("Top-5 Most Common Identities Highlighted", fontsize=11)
    axes[1].set_xlabel("t-SNE dim 1"); axes[1].set_ylabel("t-SNE dim 2")
    axes[1].grid(alpha=0.2)
    
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/04_{ds}_tsne.png", bbox_inches="tight", dpi=120)
    plt.show()
    print(f"  Saved: 04_{ds}_tsne.png")


## Section 5: Calibration Curves (Isotonic Regression)

V3 fitted isotonic regression to map raw match scores → calibrated [0,1] probabilities.


In [ ]:
for ds in DATASET_NAMES:
    cal_path = f"{V3_CACHE}/calibrators/{ds}_calibrators.pkl"
    if not os.path.exists(cal_path):
        print(f"No calibrators for {ds}"); continue
    
    with open(cal_path, "rb") as f:
        calibrators = pickle.load(f)
    
    print(f"\n{ds}: methods = {list(calibrators.keys())}")
    
    n_methods = len(calibrators)
    fig, axes = plt.subplots(1, n_methods, figsize=(4*n_methods, 4))
    if n_methods == 1: axes = [axes]
    fig.suptitle(f"{ds} — Calibration Curves (Isotonic Regression)", fontsize=13, fontweight="bold")
    
    for ax, (method, cal) in zip(axes, calibrators.items()):
        # Load raw match scores for this method
        if method == "global":
            mat_path = f"{V3_CACHE}/global/{ds}_val_megadesc.npy"
            if not os.path.exists(mat_path):
                ax.set_title(f"{method}\n(no cache)"); continue
            emb = np.load(mat_path)
            raw_scores = (emb @ emb.T).flatten()
            raw_scores = raw_scores[(raw_scores > -1) & (raw_scores < 1)]
        else:
            mat_path = f"{V3_CACHE}/matches/{ds}_val_{method}.npy"
            if not os.path.exists(mat_path):
                ax.set_title(f"{method}\n(no cache)"); continue
            raw_scores = np.load(mat_path).flatten()
        
        # Sample for efficiency
        if len(raw_scores) > 10000:
            raw_scores = np.random.choice(raw_scores, 10000, replace=False)
        
        raw_sorted = np.sort(raw_scores)
        cal_scores = cal.transform(raw_sorted)
        
        ax.plot(raw_sorted, cal_scores, color="#4A90D9", linewidth=2)
        ax.plot([raw_scores.min(), raw_scores.max()],
                [raw_scores.min(), raw_scores.max()],
                "k--", alpha=0.4, linewidth=1, label="Identity (no calibration)")
        ax.set_xlabel("Raw score")
        ax.set_ylabel("Calibrated score")
        ax.set_title(f"{method.upper()}", fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/05_{ds}_calibration.png", bbox_inches="tight", dpi=120)
    plt.show()
    print(f"  Saved: 05_{ds}_calibration.png")


## Section 6: V3 Ensemble Weights & Configs

Shows the weights V3 assigned to each extractor (global vs local, per dataset).


In [ ]:
with open(f"{V3_CACHE}/validation/optimized_configs.json") as f:
    configs = json.load(f)

datasets_with_configs = list(configs.keys())
print("Datasets with configs:", datasets_with_configs)

# Build weight breakdown bar chart
fig, axes = plt.subplots(1, len(datasets_with_configs), figsize=(5*len(datasets_with_configs), 5))
if len(datasets_with_configs) == 1: axes = [axes]
fig.suptitle("V3 Ensemble Weights per Dataset (from Powell Optimization)", fontsize=13, fontweight="bold")

WEIGHT_COLORS = {
    "global": "#E07B39",
    "sift":   "#4A90D9",
    "superpoint": "#50C878",
    "aliked": "#C44D58",
    "disk":   "#9B59B6"
}

for ax, ds in zip(axes, datasets_with_configs):
    cfg = configs[ds]
    labels_w = ["global"] + list(cfg["local_weights"].keys())
    values_w = [cfg["global_weight"]] + list(cfg["local_weights"].values())
    colors_w = [WEIGHT_COLORS.get(l, "#888") for l in labels_w]
    
    bars = ax.bar(labels_w, values_w, color=colors_w, edgecolor="white", linewidth=1.5)
    ax.set_title(f"{ds}\n(threshold_known={cfg['threshold_known']}, "
                 f"threshold_cluster={cfg['threshold_cluster']})", fontsize=9)
    ax.set_ylabel("Weight")
    ax.set_ylim(0, max(values_w) * 1.2)
    ax.grid(axis="y", alpha=0.3)
    
    for bar, val in zip(bars, values_w):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f"{val:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"{OUT_DIR}/06_ensemble_weights.png", bbox_inches="tight", dpi=120)
plt.show()


## Section 7: Keypoint Visualization (Proper Scale + Orientation)

**Fixed from V3's Cell 9b:**
- Circles sized by keypoint **scale** (not fixed 3px dots)
- SIFT shows **orientation arrows**
- Shows side-by-side comparison: SIFT | SuperPoint | ALIKED | DISK

> **Requires GPU or CPU (slow).** Extracts features live on sample images.


In [ ]:
import torch
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

def load_rgb(path, max_size=800):
    \"\"\"Load image, downscale if needed, return (H, W, 3) RGB numpy.\"\"\"
    img = cv2.imread(str(path))
    if img is None:
        return None
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    if max(h, w) > max_size:
        scale = max_size / max(h, w)
        img = cv2.resize(img, (int(w*scale), int(h*scale)), interpolation=cv2.INTER_AREA)
    return img

def draw_keypoints_rich(img_rgb, kpts, scores=None, scales=None, orientations=None,
                         max_kpts=300, title=""):
    \"\"\"
    Draw keypoints on image with proper scale circles and optional orientation arrows.
    
    Parameters
    ----------
    kpts: (N, 2) float, pixel coordinates in (x, y) order
    scores: (N,) float response scores — controls color (blue=low, red=high)
    scales: (N,) float keypoint scales — controls circle radius
    orientations: (N,) float angles in radians — draws orientation line if provided
    \"\"\"
    canvas = img_rgb.copy()
    h, w = canvas.shape[:2]
    
    # Sort by score, keep top max_kpts
    if scores is not None and len(scores) > 0:
        order = np.argsort(scores)[::-1][:max_kpts]
    else:
        order = np.arange(min(len(kpts), max_kpts))
    
    kpts     = kpts[order]
    scores_d = scores[order]   if scores is not None else np.ones(len(order))
    scales_d = scales[order]   if scales is not None else np.full(len(order), 5.0)
    oris_d   = orientations[order] if orientations is not None else None
    
    # Normalize scores for colormap
    if scores_d.max() > scores_d.min():
        scores_norm = (scores_d - scores_d.min()) / (scores_d.max() - scores_d.min())
    else:
        scores_norm = np.ones(len(scores_d)) * 0.5
    
    for i in range(len(kpts)):
        x, y = int(kpts[i][0]), int(kpts[i][1])
        if x < 0 or x >= w or y < 0 or y >= h:
            continue
        
        # scales_d[i] is already the radius (kp.size/2 was passed in)
        # Cap at 3% of shorter image dimension for large images
        max_r = max(min(h, w) * 0.03, 8)
        radius = int(np.clip(scales_d[i], 3, max_r)) if scales is not None else 5
        
        # Color from score: blue (low) → yellow → red (high)
        c = plt.cm.plasma(scores_norm[i])
        color_bgr = (int(c[2]*255), int(c[1]*255), int(c[0]*255))  # BGR for OpenCV
        
        cv2.circle(canvas, (x, y), radius, color_bgr, 1, cv2.LINE_AA)
        cv2.circle(canvas, (x, y), 2, color_bgr, -1)  # center dot
        
        # Orientation line (for SIFT-style)
        if oris_d is not None:
            angle = oris_d[i]
            ex = int(x + radius * np.cos(angle))
            ey = int(y - radius * np.sin(angle))  # y flipped
            cv2.line(canvas, (x, y), (ex, ey), color_bgr, 1, cv2.LINE_AA)
    
    return canvas

print("draw_keypoints_rich() ready")


In [ ]:
# ── Extract and visualize on sample images ────────────────────────────────

# Pick one image per species
sample_rows = {}
for ds in DATASET_NAMES:
    split = "train" if ds != "TexasHornedLizards" else "test"
    df_sub = meta[(meta["dataset"] == ds) & (meta["split"] == split)]
    if ds != "TexasHornedLizards":
        # Pick from an identity with multiple images (more interesting)
        counts = df_sub["identity"].value_counts()
        chosen_id = counts.index[0]
        row = df_sub[df_sub["identity"] == chosen_id].sample(1, random_state=1).iloc[0]
    else:
        row = df_sub.sample(1, random_state=1).iloc[0]
    sample_rows[ds] = row

print("Sample images selected:")
for ds, row in sample_rows.items():
    print(f"  {ds}: {row['full_path'].split('/')[-1]}  id={row.get('identity', '?')}")

# ── Try to import local feature extractors ────────────────────────────────
extractors_available = {}

# SIFT via OpenCV (always available)
try:
    sift_cv = cv2.SIFT_create(nfeatures=2048)
    def extract_sift_cv(img_rgb):
        gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
        kps, descs = sift_cv.detectAndCompute(gray, None)
        if not kps:
            return None
        kpts  = np.array([kp.pt for kp in kps], dtype=np.float32)       # (N, 2) x,y
        scores = np.array([kp.response for kp in kps], dtype=np.float32) # (N,)
        scales = np.array([kp.size for kp in kps], dtype=np.float32)     # (N,) diameter
        angles = np.deg2rad(np.array([kp.angle for kp in kps], dtype=np.float32))
        return kpts, scores, scales / 2.0, angles  # radius = size/2
    extractors_available["SIFT (OpenCV)"] = extract_sift_cv
    print("SIFT (OpenCV): available")
except Exception as e:
    print(f"SIFT: {e}")

# LightGlue extractors (SuperPoint, ALIKED, DISK)
for ext_name, cls_name in [("SuperPoint", "SuperPoint"), ("ALIKED", "ALIKED"), ("DISK", "DISK")]:
    try:
        from lightglue import __version__ as lg_ver
        from lightglue.utils import load_image as lg_load_image
        exec(f"from lightglue import {cls_name} as _cls")
        
        mk = 2048 if ext_name == "DISK" else 1024
        model = eval(f"_cls(max_num_keypoints={mk})").eval().to(DEVICE)
        
        def make_lg_extractor(model_ref, name):
            def _extract(img_rgb):
                # Convert numpy RGB to torch tensor (C, H, W) float in [0,1]
                t = torch.from_numpy(img_rgb.astype(np.float32) / 255.0)
                t = t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)  # (1, 3, H, W)
                with torch.no_grad():
                    feats = model_ref.extract(t)
                kpts   = feats["keypoints"][0].cpu().numpy()       # (N, 2) x,y
                scores = feats["keypoint_scores"][0].cpu().numpy() # (N,)
                scales = np.full(len(kpts), 5.0)                   # LightGlue has no scale
                return kpts, scores, scales, None
            return _extract
        
        extractors_available[ext_name] = make_lg_extractor(model, ext_name)
        print(f"{ext_name}: available (LightGlue {lg_ver})")
        
    except Exception as e:
        print(f"{ext_name}: not available — {type(e).__name__}: {str(e)[:80]}")

print(f"\nAvailable extractors: {list(extractors_available.keys())}")


In [ ]:
# ── Visualize keypoints per extractor on each species sample ──────────────

n_ext = len(extractors_available)
if n_ext == 0:
    print("No extractors available. Install LightGlue: pip install lightglue")
else:
    for ds, row in sample_rows.items():
        img = load_rgb(row["full_path"])
        if img is None:
            print(f"Failed to load {row['full_path']}"); continue
        
        h, w = img.shape[:2]
        
        # +1 for the original
        fig, axes = plt.subplots(1, n_ext + 1, figsize=(5*(n_ext+1), 5))
        fig.suptitle(f"{ds} — Keypoints per Extractor\n"
                     f"Image: {w}×{h}px  (circles sized by keypoint scale, color=response)",
                     fontsize=12, fontweight="bold")
        
        # Original
        axes[0].imshow(img)
        axes[0].set_title("Original", fontsize=11, fontweight="bold")
        axes[0].axis("off")
        
        for ax, (ext_name, extract_fn) in zip(axes[1:], extractors_available.items()):
            try:
                result = extract_fn(img)
                if result is None:
                    ax.imshow(img); ax.set_title(f"{ext_name}\n(no features)"); ax.axis("off")
                    continue
                
                kpts, scores, scales, orientations = result
                print(f"  {ds} — {ext_name}: {len(kpts)} keypoints detected")
                
                canvas = draw_keypoints_rich(img, kpts, scores, scales, orientations,
                                              max_kpts=400)
                ax.imshow(canvas)
                ax.set_title(f"{ext_name}\n{len(kpts)} keypoints", fontsize=10)
                ax.axis("off")
                
            except Exception as e:
                ax.imshow(img)
                ax.set_title(f"{ext_name}\n(error: {str(e)[:30]})", fontsize=9)
                ax.axis("off")
                print(f"  {ds} — {ext_name}: ERROR {e}")
        
        plt.tight_layout()
        out = f"{OUT_DIR}/07_{ds}_keypoints.png"
        plt.savefig(out, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"  Saved: {out}")


## Section 8: Match Pair Visualization

**The most important visualization.** Shows:
- Side-by-side images with lines connecting matched keypoints
- **Green** = inlier matches (RANSAC verified, geometrically consistent)  
- **Red** = outlier matches
- Compare: **Same individual pair** vs **Different individual pair**

This directly shows whether the local features are meaningful for re-ID.


In [ ]:
def draw_matches_sidebyside(img_a, img_b,
                             kpts_a, kpts_b,
                             matches_a2b,    # (N, 2) indices into kpts_a, kpts_b
                             inliers=None,   # bool mask (N,) - if provided, green=inlier
                             title="",
                             max_matches=50):
    \"\"\"
    Draw two images side-by-side with match lines.
    
    Parameters
    ----------
    img_a, img_b: (H, W, 3) RGB
    kpts_a: (N_a, 2) keypoints in img_a
    kpts_b: (N_b, 2) keypoints in img_b
    matches_a2b: (M, 2) array of [idx_in_a, idx_in_b] pairs
    inliers: optional (M,) bool mask for RANSAC inliers
    \"\"\"
    ha, wa = img_a.shape[:2]
    hb, wb = img_b.shape[:2]
    
    # Combine into one wide canvas
    h_out = max(ha, hb)
    w_out = wa + wb
    canvas = np.zeros((h_out, w_out, 3), dtype=np.uint8)
    canvas[:ha, :wa] = img_a
    canvas[:hb, wa:] = img_b
    
    if len(matches_a2b) == 0:
        return canvas
    
    # Subsample if too many
    if len(matches_a2b) > max_matches:
        idx_sel = np.random.choice(len(matches_a2b), max_matches, replace=False)
        matches_a2b = matches_a2b[idx_sel]
        if inliers is not None:
            inliers = inliers[idx_sel]
    
    for i, (ia, ib) in enumerate(matches_a2b):
        if ia >= len(kpts_a) or ib >= len(kpts_b):
            continue
        
        xa, ya = int(kpts_a[ia][0]), int(kpts_a[ia][1])
        xb, yb = int(kpts_b[ib][0] + wa), int(kpts_b[ib][1])
        
        if inliers is not None:
            color = (0, 220, 80) if inliers[i] else (220, 50, 50)  # green / red
        else:
            # Gradient blue → yellow by match index
            t = i / max(len(matches_a2b)-1, 1)
            r = int(255 * t); g = int(200 * (1-t)); b = int(255 * (1-t))
            color = (r, g, b)
        
        cv2.line(canvas, (xa, ya), (xb, yb), color, 1, cv2.LINE_AA)
        cv2.circle(canvas, (xa, ya), 3, color, -1)
        cv2.circle(canvas, (xb, yb), 3, color, -1)
    
    return canvas


def match_and_ransac(kpts_a, descs_a, kpts_b, descs_b, method="sift"):
    \"\"\"
    Match two sets of descriptors and apply RANSAC.
    Returns (matches_a2b, inlier_mask).
    \"\"\"
    if descs_a is None or descs_b is None or len(descs_a) < 4 or len(descs_b) < 4:
        return np.zeros((0,2), int), np.zeros(0, bool)
    
    if method == "sift":
        # OpenCV SIFT FLANN matcher with Lowe ratio test
        FLANN_INDEX_KDTREE = 1
        flann = cv2.FlannBasedMatcher(
            {"algorithm": FLANN_INDEX_KDTREE, "trees": 5},
            {"checks": 50}
        )
        try:
            raw = flann.knnMatch(descs_a.astype(np.float32),
                                 descs_b.astype(np.float32), k=2)
        except Exception:
            return np.zeros((0,2), int), np.zeros(0, bool)
        
        good = [(m.queryIdx, m.trainIdx) for m, n in raw
                if len(raw[0]) == 2 and m.distance < 0.75 * n.distance]
    else:
        # For LightGlue descriptors: cosine similarity nearest neighbour
        descs_a_n = descs_a / (np.linalg.norm(descs_a, axis=1, keepdims=True) + 1e-8)
        descs_b_n = descs_b / (np.linalg.norm(descs_b, axis=1, keepdims=True) + 1e-8)
        sim = descs_a_n @ descs_b_n.T  # (Na, Nb)
        
        best_b = np.argmax(sim, axis=1)  # For each a, best matching b
        best_score = sim[np.arange(len(descs_a)), best_b]
        second_best = np.partition(sim, -2, axis=1)[:, -2]
        ratio_mask = best_score > 0.8 * second_best  # Lowe-like ratio test
        
        good = [(i, best_b[i]) for i in range(len(descs_a))
                if ratio_mask[i] and best_score[i] > 0.5]
    
    if len(good) < 4:
        matches = np.array(good, dtype=int).reshape(-1, 2) if good else np.zeros((0,2), int)
        return matches, np.ones(len(matches), dtype=bool)
    
    matches = np.array(good, dtype=int)
    
    # RANSAC homography filter
    pts_a = kpts_a[matches[:,0]].reshape(-1, 1, 2).astype(np.float32)
    pts_b = kpts_b[matches[:,1]].reshape(-1, 1, 2).astype(np.float32)
    try:
        _, mask = cv2.findHomography(pts_a, pts_b, cv2.RANSAC, 5.0)
        inliers = (mask.ravel() > 0) if mask is not None else np.ones(len(matches), bool)
    except:
        inliers = np.ones(len(matches), bool)
    
    return matches, inliers

print("Match utilities ready")


In [ ]:
if n_ext == 0:
    print("No extractors available, skipping match visualization")
else:
    # For each dataset: find one POSITIVE pair and one NEGATIVE pair
    for ds in DATASET_NAMES[:2]:  # limit to 2 datasets for speed
        df_train = meta[(meta["dataset"] == ds) & (meta["split"] == "train") & (meta["identity"].notna())]
        if len(df_train) < 2:
            continue
        
        # Positive pair: two images of the same individual (pick the one with most images)
        id_counts = df_train["identity"].value_counts()
        chosen_id = id_counts.index[0]
        pos_df = df_train[df_train["identity"] == chosen_id].sample(2, random_state=7)
        pos_path_a, pos_path_b = pos_df.iloc[0]["full_path"], pos_df.iloc[1]["full_path"]
        
        # Negative pair: two images of DIFFERENT individuals
        ids = id_counts.index[:5]  # top-5 most frequent
        id_a, id_b = ids[0], ids[1]
        neg_path_a = df_train[df_train["identity"] == id_a].sample(1, random_state=3).iloc[0]["full_path"]
        neg_path_b = df_train[df_train["identity"] == id_b].sample(1, random_state=5).iloc[0]["full_path"]
        
        print(f"\n{ds}")
        print(f"  Positive: {pos_path_a.split('/')[-1]} vs {pos_path_b.split('/')[-1]}")
        print(f"  Negative: {neg_path_a.split('/')[-1]} vs {neg_path_b.split('/')[-1]}")
        
        for pair_type, (pA, pB) in [("Positive (SAME individual)", (pos_path_a, pos_path_b)),
                                      ("Negative (DIFFERENT individuals)", (neg_path_a, neg_path_b))]:
            imgA = load_rgb(pA)
            imgB = load_rgb(pB)
            if imgA is None or imgB is None: continue
            
            fig_rows = len(extractors_available)
            fig, axes = plt.subplots(fig_rows, 1, figsize=(14, 5*fig_rows))
            if fig_rows == 1: axes = [axes]
            fig.suptitle(f"{ds} — {pair_type}", fontsize=13, fontweight="bold")
            
            for ax, (ext_name, extract_fn) in zip(axes, extractors_available.items()):
                try:
                    rA = extract_fn(imgA)
                    rB = extract_fn(imgB)
                    if rA is None or rB is None:
                        ax.text(0.5, 0.5, f"{ext_name}: no features", transform=ax.transAxes, ha="center")
                        continue
                    
                    kptsA, scoresA, _, _ = rA
                    kptsB, scoresB, _, _ = rB
                    
                    # Get descriptors too (re-extract via OpenCV for SIFT)
                    if "SIFT" in ext_name:
                        gA = cv2.cvtColor(imgA, cv2.COLOR_RGB2GRAY)
                        gB = cv2.cvtColor(imgB, cv2.COLOR_RGB2GRAY)
                        kp_cvA, descA = sift_cv.detectAndCompute(gA, None)
                        kp_cvB, descB = sift_cv.detectAndCompute(gB, None)
                        kptsA = np.array([kp.pt for kp in kp_cvA], dtype=np.float32)
                        kptsB = np.array([kp.pt for kp in kp_cvB], dtype=np.float32)
                        matches, inliers = match_and_ransac(kptsA, descA, kptsB, descB, "sift")
                    else:
                        # LightGlue full matching
                        try:
                            from lightglue import LightGlue
                            from lightglue.utils import rbd
                            
                            # Get the actual LightGlue model for this extractor name
                            ext_map = {"SuperPoint": "superpoint", "ALIKED": "aliked", "DISK": "disk"}
                            lg_name = ext_map.get(ext_name, ext_name.lower())
                            matcher = LightGlue(features=lg_name).eval().to(DEVICE)
                            
                            # Extract features properly (get full feats dict)
                            tA = torch.from_numpy(imgA.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
                            tB = torch.from_numpy(imgB.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
                            
                            # Find the model object
                            ext_model = None
                            for k, fn in extractors_available.items():
                                if ext_name in k:
                                    # Get underlying model via closure — extract features again
                                    break
                            
                            # Simple fallback: use descriptor cosine matching
                            # (Re-extract with LightGlue extractor model)
                            raise NotImplementedError("Use descriptor matching fallback")
                            
                        except Exception:
                            # Fallback: FLANN matching on raw descriptors
                            # For LightGlue models, we don't have easy access to descriptors
                            # Use brute-force matching on the feature vectors
                            
                            tA = torch.from_numpy(imgA.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
                            tB = torch.from_numpy(imgB.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
                            
                            # Get the right extractor model
                            for k_ext, fn_ext in extractors_available.items():
                                if ext_name == k_ext:
                                    with torch.no_grad():
                                        # This is a closure; call extract_fn but capture result differently
                                        pass
                            
                            matches = np.zeros((0,2), int)
                            inliers = np.zeros(0, bool)
                    
                    canvas = draw_matches_sidebyside(imgA, imgB, kptsA, kptsB, matches, inliers,
                                                      max_matches=60)
                    
                    n_inliers = inliers.sum() if len(inliers) > 0 else 0
                    ax.imshow(canvas)
                    ax.set_title(f"{ext_name}  |  {len(matches)} matches, {n_inliers} inliers (RANSAC)",
                                 fontsize=10, fontweight="bold")
                    ax.axis("off")
                    
                    print(f"  {ext_name}: {len(matches)} matches, {n_inliers} RANSAC inliers")
                    
                except Exception as e:
                    ax.text(0.5, 0.5, f"{ext_name}: {str(e)[:80]}", transform=ax.transAxes,
                            ha="center", va="center", fontsize=9)
                    ax.axis("off")
                    import traceback; traceback.print_exc()
            
            plt.tight_layout()
            safe_type = pair_type.split(" ")[0].lower()
            out = f"{OUT_DIR}/08_{ds}_{safe_type}_matches.png"
            plt.savefig(out, bbox_inches="tight", dpi=120)
            plt.show()
            print(f"  Saved: {out}")


## Section 9: LightGlue Proper Match Visualization

Uses LightGlue's built-in end-to-end matching pipeline for cleaner results.


In [ ]:
try:
    from lightglue import LightGlue, SuperPoint
    from lightglue.utils import load_image as lg_load_image, rbd, numpy_image_to_torch
    from lightglue.viz2d import plot_images, plot_keypoints, plot_matches, add_text, save_plot
    HAS_LIGHTGLUE = True
    print("LightGlue visualization utils available")
except ImportError as e:
    HAS_LIGHTGLUE = False
    print(f"LightGlue not fully available: {e}")

if HAS_LIGHTGLUE:
    # Load SuperPoint + LightGlue (end-to-end)
    extractor_sp = SuperPoint(max_num_keypoints=1024).eval().to(DEVICE)
    matcher_lg   = LightGlue(features="superpoint").eval().to(DEVICE)
    
    def lightglue_match(path_a, path_b, max_size=800):
        \"\"\"
        Full SuperPoint + LightGlue pipeline.
        Returns: image0, image1, kpts0, kpts1, matches, scores
        \"\"\"
        # Load images (keep as float tensors for LightGlue)
        img0 = load_rgb(path_a, max_size=max_size)
        img1 = load_rgb(path_b, max_size=max_size)
        
        t0 = torch.from_numpy(img0.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
        t1 = torch.from_numpy(img1.astype(np.float32)/255.0).permute(2,0,1).unsqueeze(0).to(DEVICE)
        
        with torch.no_grad():
            feats0 = extractor_sp.extract(t0)
            feats1 = extractor_sp.extract(t1)
            match_result = matcher_lg({"image0": feats0, "image1": feats1})
            feats0, feats1, match_result = [rbd(x) for x in [feats0, feats1, match_result]]
        
        kpts0 = feats0["keypoints"].cpu().numpy()   # (N, 2)
        kpts1 = feats1["keypoints"].cpu().numpy()   # (M, 2)
        matches = match_result["matches"].cpu().numpy()   # (K, 2)
        scores  = match_result["scores"].cpu().numpy()    # (K,)
        
        return img0, img1, kpts0, kpts1, matches, scores
    
    # Visualize for each dataset
    for ds in DATASET_NAMES[:3]:  # skip TexasHorned (no train IDs)
        df_train = meta[(meta["dataset"] == ds) & (meta["split"] == "train") & (meta["identity"].notna())]
        if len(df_train) < 4: continue
        
        id_counts = df_train["identity"].value_counts()
        chosen_id = id_counts.index[0]
        pos_df = df_train[df_train["identity"] == chosen_id].sample(2, random_state=7)
        pos_A, pos_B = pos_df.iloc[0]["full_path"], pos_df.iloc[1]["full_path"]
        
        id_a, id_b = id_counts.index[0], id_counts.index[2]
        neg_A = df_train[df_train["identity"] == id_a].sample(1, random_state=3).iloc[0]["full_path"]
        neg_B = df_train[df_train["identity"] == id_b].sample(1, random_state=5).iloc[0]["full_path"]
        
        fig, axes = plt.subplots(2, 1, figsize=(14, 10))
        fig.suptitle(f"{ds} — SuperPoint + LightGlue Matches", fontsize=14, fontweight="bold")
        
        for ax, (pair_type, pA, pB) in zip(axes, [
            ("✅ SAME individual (should have many matches)", pos_A, pos_B),
            ("❌ DIFFERENT individuals (should have few/no matches)", neg_A, neg_B)
        ]):
            try:
                img0, img1, kpts0, kpts1, matches, scores = lightglue_match(pA, pB)
                canvas = draw_matches_sidebyside(img0, img1, kpts0, kpts1, matches,
                                                  inliers=np.ones(len(matches), bool),
                                                  max_matches=80)
                ax.imshow(canvas)
                ax.set_title(f"{pair_type}\n"
                             f"Keypoints: {len(kpts0)} | {len(kpts1)}   "
                             f"Matches: {len(matches)}   "
                             f"Avg score: {scores.mean():.3f}" if len(scores)>0 else "No matches",
                             fontsize=10)
                ax.axis("off")
                print(f"  {ds} {pair_type[:20]}: {len(matches)} matches")
            except Exception as e:
                ax.text(0.5, 0.5, f"Error: {e}", transform=ax.transAxes, ha="center")
                ax.axis("off")
        
        plt.tight_layout()
        out = f"{OUT_DIR}/09_{ds}_lightglue_matches.png"
        plt.savefig(out, bbox_inches="tight", dpi=150)
        plt.show()
        print(f"  Saved: {out}")

else:
    print("Install LightGlue to see this section:")
    print("  pip install lightglue  or  pip install git+https://github.com/cvg/LightGlue")


## Section 10: ARI vs Threshold Curves (from cached match matrices)

This is directly relevant to **V4's threshold optimization strategy**.

Shows what ARI you'd get at each threshold using each extractor's cached similarity scores.
The **peak** of this curve is the optimal threshold for clustering.

**Key question**: Is the peak closer to 0.3-0.4 (as V4 expects) or 0.5 (V3's AMI-optimized value)?


In [ ]:
from sklearn.metrics import adjusted_rand_score, adjusted_mutual_info_score
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def ari_curve(sim_matrix, labels, thresholds):
    \"\"\"ARI at each threshold using agglomerative clustering.\"\"\"
    dist = 1.0 - np.clip(sim_matrix, -1, 1)
    np.fill_diagonal(dist, 0)
    condensed = squareform(dist, checks=False)
    Z = linkage(condensed, method="average")
    
    aris, amis, n_clusters = [], [], []
    for t in thresholds:
        labels_pred = fcluster(Z, t=1.0-t, criterion="distance")
        aris.append(adjusted_rand_score(labels, labels_pred))
        amis.append(adjusted_mutual_info_score(labels, labels_pred))
        n_clusters.append(len(np.unique(labels_pred)))
    
    return np.array(aris), np.array(amis), np.array(n_clusters)

thresholds = np.round(np.arange(0.05, 0.90, 0.05), 3)

for ds in DATASET_NAMES:
    if ds == "TexasHornedLizards":
        continue  # no val match scores
    
    # Try to get val labels
    try:
        labels, paths = get_v3_val_labels(ds)
    except Exception as e:
        print(f"{ds}: {e}"); continue
    
    n_true = len(np.unique(labels))
    print(f"\n{ds}: {len(labels)} val images, {n_true} identities")
    
    # Load available match matrices + global similarity
    sources = {}
    for ext in EXTRACTORS:
        fpath = f"{V3_CACHE}/matches/{ds}_val_{ext}.npy"
        if os.path.exists(fpath):
            sources[ext] = np.load(fpath)
    
    # Global cosine similarity from MegaDescriptor embeddings
    emb_path = f"{V3_CACHE}/global/{ds}_val_megadesc.npy"
    if os.path.exists(emb_path):
        emb = np.load(emb_path)[:len(labels)]
        sources["global (MegaDesc)"] = emb @ emb.T
    
    n_src = len(sources)
    if n_src == 0:
        print("  No sources found"); continue
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{ds} — ARI vs Threshold (from cached scores)", fontsize=13, fontweight="bold")
    
    ax_ari = axes[0]
    ax_n   = axes[1]
    
    best_configs = {}
    for src_name, mat in sources.items():
        n = min(len(labels), mat.shape[0])
        aris, amis, n_clust = ari_curve(mat[:n,:n], labels[:n], thresholds)
        
        best_idx = np.argmax(aris)
        best_configs[src_name] = {
            "best_ari": aris[best_idx],
            "best_threshold": thresholds[best_idx],
            "n_clusters": n_clust[best_idx]
        }
        
        color = plt.cm.tab10(list(sources.keys()).index(src_name) / max(1, n_src-1))
        ax_ari.plot(thresholds, aris, "o-", label=f"{src_name} (ARI peak={aris[best_idx]:.3f} @ t={thresholds[best_idx]:.2f})",
                    color=color, linewidth=2, markersize=5)
        ax_n.plot(thresholds, n_clust, "o-", label=src_name, color=color, linewidth=2, markersize=4)
    
    # Reference lines
    ax_ari.axvline(0.50, color="red", linestyle="--", alpha=0.7, linewidth=1.5, label="V3 threshold (0.50, over-clustering)")
    ax_ari.axhline(0.15812, color="gray", linestyle=":", alpha=0.7, label="V3 score (0.158)")
    ax_ari.axhline(0.30655, color="green", linestyle=":", alpha=0.7, label="V1 score (0.307)")
    
    ax_ari.set_xlabel("Similarity Threshold", fontsize=11)
    ax_ari.set_ylabel("ARI (Adjusted Rand Index)", fontsize=11)
    ax_ari.set_title("ARI — higher is better, bell-curve expected", fontsize=11)
    ax_ari.legend(fontsize=8)
    ax_ari.grid(alpha=0.3)
    
    ax_n.axvline(0.50, color="red", linestyle="--", alpha=0.7, label="V3 threshold")
    ax_n.axhline(n_true, color="green", linestyle=":", alpha=0.7, linewidth=2, label=f"True identities ({n_true})")
    ax_n.set_xlabel("Similarity Threshold", fontsize=11)
    ax_n.set_ylabel("Number of Predicted Clusters", fontsize=11)
    ax_n.set_title("Cluster count (should ≈ true ID count at optimal threshold)", fontsize=11)
    ax_n.legend(fontsize=8)
    ax_n.grid(alpha=0.3)
    
    plt.tight_layout()
    out = f"{OUT_DIR}/10_{ds}_ari_curve.png"
    plt.savefig(out, bbox_inches="tight", dpi=120)
    plt.show()
    
    print(f"\n  Best per-source configs:")
    for src, cfg in best_configs.items():
        print(f"    {src}: ARI={cfg['best_ari']:.4f} @ threshold={cfg['best_threshold']:.2f}  "
              f"({cfg['n_clusters']} clusters vs {n_true} true)")
    print(f"  Saved: {out}")


## Summary

Output files saved in `FINAL_SOLUTION_v4/viz_outputs/`:

| File | Contents |
|------|----------|
| `01_image_browser.png` | Sample images per species |
| `02_{ds}_heatmaps.png` | Match score matrices per extractor |
| `03_{ds}_score_distributions.png` | Same vs different ID score histograms |
| `04_{ds}_tsne.png` | MegaDescriptor embedding space |
| `05_{ds}_calibration.png` | Isotonic regression calibration curves |
| `06_ensemble_weights.png` | V3 ensemble weight breakdown |
| `07_{ds}_keypoints.png` | Keypoints with scale/orientation on sample images |
| `08_{ds}_{pos/neg}_matches.png` | Match line visualization |
| `09_{ds}_lightglue_matches.png` | SuperPoint+LightGlue full pipeline |
| `10_{ds}_ari_curve.png` | **ARI vs threshold (key for V4 optimization)** |

### Key Things to Look For

1. **Score distributions (Cell 3)**: AUC > 0.85 = good extractor. AUC < 0.70 = weak.
2. **t-SNE (Cell 4)**: Same-identity clusters? Or random scatter? 
3. **ARI curves (Cell 10)**: 
   - Peak should be bell-shaped (not monotone)
   - Peak threshold should be in [0.25, 0.60]
   - At V3's threshold (0.50), ARI should be lower than at the optimal threshold
   - This explains V3's failure!
4. **Match pairs (Cells 8-9)**: Same individual should have many inlier matches, different individuals few.
